In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wave
!pip install python_speech_features
from python_speech_features import logfbank
import os

  Preparing metadata (setup.py) ... done
  Created wheel for python_speech_features: filename=python_speech_features-0.6-py3-none-any.whl size=5868 sha256=3efd862fcf8d90a80e97e02f279b61d68231d48d26663355dffca8013ed3af4c
  Stored in directory: /root/.cache/pip/wheels/60/90/3c/4b5996a95d363fa14525597a19146a940bec467b44b2a14580
Successfully built python_speech_features


In [ ]:
# --- Colab Setup ---
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ----- Helper functions -----
def read_wav(filepath):
    """Read a 16-bit mono WAV file."""
    with wave.open(filepath, "rb") as wf:
        frames = wf.readframes(wf.getnframes())
        sig = np.frombuffer(frames, dtype=np.int16).astype(np.float32)
        sr = wf.getframerate()
    return sig, sr

def compute_mel(filepath, nfilts=26):
    """Compute mel filterbank energies."""
    sig, sr = read_wav(filepath)
    feat = logfbank(sig, sr, nfilt=nfilts, nfft=512)
    feat = np.flipud(feat.T)  # low freq at bottom
    return feat

In [ ]:
# ----- Grid Plot Function -----
def plot_mel_grid_colab(dataset_dir, labels=["w","o","n"], n_examples=3, nfilts=24):
    """
    Plot a grid of mel spectrograms from a Google Drive folder.
    - n_examples: number of examples per label
    """
    n_labels = len(labels)
    fig, axes = plt.subplots(n_labels, n_examples, figsize=(n_examples*3, n_labels*2.5))

    for i, label in enumerate(labels):
        label_dir = os.path.join(dataset_dir, label)
        if not os.path.exists(label_dir):
            print(f"Label folder not found: {label_dir}")
            continue

        files = [f for f in os.listdir(label_dir) if f.endswith(".wav")]
        files = files[:n_examples]

        for j, f in enumerate(files):
            feat = compute_mel(os.path.join(label_dir, f), nfilts=nfilts)
            ax = axes[i, j] if n_labels > 1 else axes[j]
            im = ax.imshow(feat, aspect='auto', origin='lower', cmap='magma')
            ax.set_xticks([])
            ax.set_yticks([])
            if j == 0:
                ax.set_ylabel(label, fontsize=10)
            ax.set_title(os.path.splitext(f)[0], fontsize=8)

    fig.tight_layout()
    cbar = fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.02, pad=0.02)
    cbar.set_label("log energy")
    plt.show()

In [ ]:
# Adjust this path to the folder in your Google Drive
dataset_dir = "/content/drive/MyDrive/wake_word_dataset_taset_oceanpanda667"
plot_mel_grid_colab(dataset_dir, labels=["w","o","n"], n_examples=3)